# Gold — GA4 page sessions (URL grain)

Python 3.12 notebook (not Spark). Default lakehouse: **GA4Lakehouse** in workspace **GA4-Report**.

Reads `dbo.bronze_ga4_events` through deltalake (storage bearer token in Fabric; not the raw Files parquet) and DuckDB SQL. Writes `dbo.gold_ga4_page_sessions` with overwrite.

**Grain:** `event_date` (DATE) + `page_location` (URL as given). `page_path` is the same URL with the scheme, host, query string, and fragment removed.

`ga_session_id` and `session_engaged` are coalesced from `event_params` `string_value` or `int_value`. Entrances and exits use each session's first and last `page_view` by `event_timestamp` (the `entrances` param is not always present). Engagement time is the sum of `engagement_time_msec` on events for that page, divided by sessions, in seconds.

Physical path: `/lakehouse/default/Tables/dbo/gold_ga4_page_sessions` (or the equivalent `abfss://` URI).


In [ ]:
# Build dbo.gold_ga4_page_sessions from dbo.bronze_ga4_events (overwrite).
import os
import subprocess
import sys
from pathlib import Path

import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

ROOT = Path(os.environ.get("LAKEHOUSE_DEFAULT", "/lakehouse/default"))
SOURCE = "bronze_ga4_events"
TABLE = "gold_ga4_page_sessions"


def _storage():
    """abfss + bearer token in Fabric; local filesystem path otherwise."""
    try:
        import notebookutils
        ctx = notebookutils.runtime.context
        ws = ctx["defaultLakehouseWorkspaceId"]
        lh = ctx["defaultLakehouseId"]
        endpoint = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
        base = f"abfss://{ws}@{endpoint}/{lh}/Tables/dbo"
        opts = {
            "bearer_token": notebookutils.credentials.getToken("storage"),
            "use_fabric_endpoint": "true",
        }
        return base, opts
    except Exception:
        dbo = ROOT / "Tables" / "dbo"
        dbo.mkdir(parents=True, exist_ok=True)
        return str(dbo), None


def _cast_null_type(typ: pa.DataType) -> pa.DataType:
    if pa.types.is_null(typ):
        return pa.string()
    if pa.types.is_struct(typ):
        return pa.struct([
            pa.field(f.name, _cast_null_type(f.type), nullable=f.nullable)
            for f in typ
        ])
    if pa.types.is_list(typ) or pa.types.is_large_list(typ) or pa.types.is_fixed_size_list(typ):
        return pa.list_(_cast_null_type(typ.value_type))
    if pa.types.is_map(typ):
        return pa.map_(_cast_null_type(typ.key_type), _cast_null_type(typ.item_type))
    return typ


def _cast_nulls(table: pa.Table) -> pa.Table:
    new_fields, new_cols = [], []
    for i, field in enumerate(table.schema):
        new_type = _cast_null_type(field.type)
        col = table.column(i)
        if new_type != field.type:
            col = col.cast(new_type)
        new_fields.append(pa.field(field.name, new_type, nullable=field.nullable))
        new_cols.append(col)
    return pa.Table.from_arrays(new_cols, schema=pa.schema(new_fields))



def _arrow(con, sql):
    rel = con.sql(sql)
    if hasattr(rel, "to_arrow_table"):
        return rel.to_arrow_table()
    if hasattr(rel, "fetch_arrow_table"):
        return rel.fetch_arrow_table()
    return rel.arrow()


def _one(con, sql):
    rows = con.sql(sql).fetchall()
    return rows[0] if rows else None


base, opts = _storage()
src_uri = f"{base}/{SOURCE}"
dst_uri = f"{base}/{TABLE}"
print(f"source: {src_uri}")
print(f"write target: {dst_uri}  (schema dbo)")

bronze_ds = DeltaTable(src_uri, storage_options=opts).to_pyarrow_dataset()
con = duckdb.connect()
con.register("bronze_ga4_events", bronze_ds)
gold = _arrow(con, """
WITH ev0 AS (
    SELECT
        strptime(CAST(event_date AS VARCHAR), '%Y%m%d')::DATE AS event_date,
        event_timestamp,
        event_name,
        user_pseudo_id,
        event_params,
        row_number() OVER () AS _rn
    FROM bronze_ga4_events
),
ev AS (
    SELECT
        ev0._rn,
        ev0.event_date,
        ev0.event_timestamp,
        ev0.event_name,
        ev0.user_pseudo_id,
        MAX(CASE WHEN x.key = 'page_location' THEN COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR)) END) AS page_location,
        MAX(CASE WHEN x.key = 'page_title' THEN COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR)) END) AS page_title,
        MAX(CASE WHEN x.key = 'ga_session_id' THEN COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR)) END) AS ga_session_id,
        MAX(CASE WHEN x.key = 'session_engaged' THEN COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR)) END) AS session_engaged,
        MAX(CASE WHEN x.key = 'engagement_time_msec' THEN COALESCE(x.value.int_value, TRY_CAST(x.value.string_value AS BIGINT)) END) AS engagement_time_msec
    FROM ev0
    LEFT JOIN LATERAL UNNEST(ev0.event_params) AS u(x) ON TRUE
    GROUP BY ev0._rn, ev0.event_date, ev0.event_timestamp, ev0.event_name, ev0.user_pseudo_id
),
page_views AS (
    SELECT * FROM ev WHERE event_name = 'page_view'
),
session_bounds AS (
    SELECT
        user_pseudo_id,
        ga_session_id,
        arg_min(page_location, event_timestamp) AS first_page,
        arg_min(event_date, event_timestamp) AS first_date,
        arg_max(page_location, event_timestamp) AS last_page,
        arg_max(event_date, event_timestamp) AS last_date
    FROM page_views
    WHERE page_location IS NOT NULL
    GROUP BY user_pseudo_id, ga_session_id
),
engaged AS (
    SELECT DISTINCT user_pseudo_id, ga_session_id
    FROM ev
    WHERE session_engaged = '1'
),
engagement AS (
    SELECT event_date, page_location, SUM(engagement_time_msec) AS engagement_time_msec
    FROM ev
    WHERE page_location IS NOT NULL
    GROUP BY 1, 2
)
SELECT
    pv.event_date,
    pv.page_location,
    COALESCE(
        NULLIF(
            regexp_replace(
                regexp_replace(pv.page_location, '[?#].*$', ''),
                '^[a-zA-Z][a-zA-Z0-9+.-]*://[^/]+',
                ''
            ),
            ''
        ),
        '/'
    ) AS page_path,
    mode(pv.page_title) AS page_title,
    COUNT(*)::BIGINT AS pageviews,
    COUNT(DISTINCT pv.user_pseudo_id || '|' || pv.ga_session_id)::BIGINT AS sessions,
    COUNT(DISTINCT pv.user_pseudo_id)::BIGINT AS users,
    COUNT(DISTINCT CASE
        WHEN sb.first_page = pv.page_location AND sb.first_date = pv.event_date
        THEN pv.user_pseudo_id || '|' || pv.ga_session_id
    END)::BIGINT AS entrances,
    COUNT(DISTINCT CASE
        WHEN eg.user_pseudo_id IS NOT NULL
        THEN pv.user_pseudo_id || '|' || pv.ga_session_id
    END)::BIGINT AS engaged_sessions,
    (COALESCE(en.engagement_time_msec, 0)::DOUBLE
        / NULLIF(COUNT(DISTINCT pv.user_pseudo_id || '|' || pv.ga_session_id), 0)
        / 1000.0) AS avg_engagement_time_sec,
    COUNT(DISTINCT CASE
        WHEN sb.last_page = pv.page_location AND sb.last_date = pv.event_date
        THEN pv.user_pseudo_id || '|' || pv.ga_session_id
    END)::BIGINT AS exits
FROM page_views pv
LEFT JOIN session_bounds sb
    ON sb.user_pseudo_id IS NOT DISTINCT FROM pv.user_pseudo_id
   AND sb.ga_session_id IS NOT DISTINCT FROM pv.ga_session_id
LEFT JOIN engaged eg
    ON eg.user_pseudo_id IS NOT DISTINCT FROM pv.user_pseudo_id
   AND eg.ga_session_id IS NOT DISTINCT FROM pv.ga_session_id
LEFT JOIN engagement en
    ON en.event_date = pv.event_date
   AND en.page_location = pv.page_location
WHERE pv.page_location IS NOT NULL
GROUP BY pv.event_date, pv.page_location, en.engagement_time_msec
ORDER BY pv.event_date, sessions DESC, pv.page_location
""")

write_deltalake(
    dst_uri,
    _cast_nulls(gold),
    mode="overwrite",
    schema_mode="overwrite",
    storage_options=opts,
)

gold_ds = DeltaTable(dst_uri, storage_options=opts).to_pyarrow_dataset()
con.register("gold_ga4_page_sessions", gold_ds)

bronze_pv, null_loc_pv = _one(con, """
SELECT
    COUNT(*)::BIGINT,
    COUNT(*) FILTER (
        WHERE (
            SELECT COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR))
            FROM UNNEST(event_params) AS u(x)
            WHERE x.key = 'page_location'
            LIMIT 1
        ) IS NULL
    )::BIGINT
FROM bronze_ga4_events
WHERE event_name = 'page_view'
""")
gold_rows, gold_pageviews, gold_null_loc = _one(con, """
SELECT
    COUNT(*)::BIGINT,
    COALESCE(SUM(pageviews), 0)::BIGINT,
    COUNT(*) FILTER (WHERE page_location IS NULL)::BIGINT
FROM gold_ga4_page_sessions
""")

print(f"bronze page_view events: {bronze_pv:,}")
print(f"gold pageviews: {gold_pageviews:,}")
print(f"page_view events with null page_location: {null_loc_pv:,}")
print(f"gold rows with null page_location: {gold_null_loc:,}")
print(f"gold rows: {gold_rows:,}")
print("top 10 pages by sessions:")
con.sql("""
SELECT
    page_location,
    ANY_VALUE(page_path) AS page_path,
    SUM(pageviews)::BIGINT AS pageviews,
    SUM(sessions)::BIGINT AS sessions,
    SUM(users)::BIGINT AS users,
    SUM(entrances)::BIGINT AS entrances,
    SUM(exits)::BIGINT AS exits
FROM gold_ga4_page_sessions
GROUP BY page_location
ORDER BY sessions DESC, pageviews DESC, page_location
LIMIT 10
""").show()

errors = []
if gold_pageviews + null_loc_pv != bronze_pv:
    errors.append(
        f"gold pageviews {gold_pageviews} + null page_location page_views {null_loc_pv} "
        f"!= bronze page_view events {bronze_pv}"
    )
if gold_null_loc:
    errors.append(f"gold has {gold_null_loc} null page_location rows")
if errors:
    msg = "; ".join(errors)
    print(f"FAIL {TABLE}: {msg}")
    raise SystemExit(f"FAIL {TABLE}: {msg}")
print(f"PASS {TABLE}: rows={gold_rows:,} pageviews={gold_pageviews:,}")
